In [1]:
!pip install datasets pandas numpy scikit-learn

In [2]:
!pip install --upgrade datasets huggingface_hub pandas

In [3]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")

C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\KIIT0001\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [10]:
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [12]:
dataset["train"][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'labels': [27],
 'id': 'eebbqej'}

In [13]:
label_names = dataset["train"].features["labels"].feature.names
print(label_names)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [14]:
target_emotions = ["joy", "sadness", "anger", "fear", "neutral"]

In [15]:
def process_example(example):
    labels = example["labels"]
    text = example["text"]
    
    multi_hot = [0]*len(target_emotions)
    
    for l in labels:
        emotion = label_names[l]
        if emotion in target_emotions:
            idx = target_emotions.index(emotion)
            multi_hot[idx] = 1
    
    return {
        "text": text,
        "emotion_vector": multi_hot
    }

In [17]:
processed_dataset = dataset.map(
    process_example,
    batched=False,
    load_from_cache_file=False,
    keep_in_memory=True
)

In [19]:
processed_dataset = processed_dataset.remove_columns(["labels", "id"])

In [20]:
processed_dataset["train"][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'emotion_vector': [0, 0, 0, 0, 1]}

In [21]:
def filter_empty(example):
    return sum(example["emotion_vector"]) > 0

processed_dataset = processed_dataset.filter(filter_empty)

In [22]:
len(processed_dataset["train"])

18906

In [23]:
def map_stress(emotion_vector):
    joy, sadness, anger, fear, neutral = emotion_vector
    
    if joy == 1 or neutral == 1:
        return 0   # low stress
    elif sadness == 1:
        return 1   # medium stress
    elif anger == 1 or fear == 1:
        return 2   # high stress
    else:
        return 1

In [24]:
def add_stress(example):
    example["stress"] = map_stress(example["emotion_vector"])
    return example

processed_dataset = processed_dataset.map(add_stress)

In [25]:
processed_dataset["train"][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'emotion_vector': [0, 0, 0, 0, 1],
 'stress': 0}

In [26]:
import pandas as pd

df = pd.DataFrame(processed_dataset["train"])
df.head()

,text,emotion_vector,stress
0,My favourite food is anything I didn't have to...,"[0, 0, 0, 0, 1]",0
1,"Now if he does off himself, everyone will thin...","[0, 0, 0, 0, 1]",0
2,WHY THE FUCK IS BAYLESS ISOING,"[0, 0, 1, 0, 0]",2
3,To make her feel threatened,"[0, 0, 0, 1, 0]",2
4,It might be linked to the trust factor of your...,"[0, 0, 0, 0, 1]",0


In [27]:
df["stress"].value_counts()

stress
0    15626
2     2025
1     1255
Name: count, dtype: int64

In [28]:
df.to_csv("emotion_dataset.csv", index=False)